# Entity-swap relation 0 ablation: full vs suppress-only vs inject-only

This Colab notebook runs the entity-swap steering experiment for BATS relation 0 (`capital_country`) on the labeled "ours" summary graphs, comparing three conditions on the **same 50 sampled source→donor pairs**:

- **full**: suppress the source Output CLT features (factor `-2`) and inject the donor Output CLT features (factor in `2,4,8`).
- **suppress-only**: suppress the source only (no donor injection).
- **inject-only**: inject the donor only (no source suppression).

The labeled summary graphs and the `--mode` eval hook both live on the `clean_up` branch, so this notebook only needs to clone the repo, load the replacement model once with the matching `mntss/clt-gemma-2-2b-426k` transcoder set, and run the three conditions. It reports the same metrics as the prior runs (Top-1 / Top-5 hit rates, success, Δp_source, Δp_donor).

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; CPU will be slow for entity-swap interventions.")

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/IamKrill1n/circuit_tracer_mod.git"
REPO_NAME = "circuit_tracer_mod"
REPO_REF = "clean_up"


def find_repo_root() -> Path | None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates.extend([Path("/home/tu/circuit_tracer_mod"), Path("/content") / REPO_NAME])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "summarization").is_dir():
            return candidate
    return None


REPO_ROOT = find_repo_root()
clone_parent = Path("/content") if Path("/content").exists() else Path.cwd()
if REPO_ROOT is None:
    subprocess.run(["git", "clone", REPO_URL, str(clone_parent / REPO_NAME)], check=True)
    REPO_ROOT = clone_parent / REPO_NAME

# Only fetch/checkout when this is a cloned Colab checkout, not the local working copy.
if REPO_ROOT == clone_parent / REPO_NAME and (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=REPO_ROOT, check=True)

print("repo root:", REPO_ROOT)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, text=True).strip())

In [ ]:
# Make repo imports work in this kernel, installing editable only on a fresh /content runtime.
repo_root_str = str(REPO_ROOT)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

try:
    import circuit_tracer  # noqa: F401
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
    if repo_root_str not in sys.path:
        sys.path.insert(0, repo_root_str)

# The --mode hook must be present in this checkout; push the clean_up branch first if not.
from eval.eval_entity_swap import build_parser

mode_action = next((a for a in build_parser()._actions if a.dest == "mode"), None)
assert mode_action is not None, "This checkout is missing the --mode eval hook. Push the clean_up branch first."
assert set(mode_action.choices) == {"full", "suppress", "inject"}, mode_action.choices
print("entity-swap --mode hook is available:", mode_action.choices)

In [ ]:
import torch

RELATION_IDX = 0
RELATION_NAME = "capital_country"
SAMPLE_PAIRS_PER_RELATION = 50
RANDOM_STATE = 42
NEG_COEFFS = "-2"
ADD_COEFFS = "2,4,8"
LAYERS_BELOW = 0
LAYERS_ABOVE = 1

MODEL_NAME = "google/gemma-2-2b"
# Must match the transcoder set the graphs were built with (feature indices are set-specific).
TRANSCODER_SET = "mntss/clt-gemma-2-2b-426k"
BACKEND = "transformerlens"
DTYPE = "bfloat16"
DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"
DTYPE_MAP = {"float32": torch.float32, "float16": torch.float16, "bfloat16": torch.bfloat16}

GRAPH_SRC_DIR = (
    REPO_ROOT
    / "summary_graphs"
    / "analogies"
    / "mntss"
    / "clt-gemma-2-2b-426k"
    / "entmax"
    / "alpha_0.50"
    / "node_0.02"
)
ANALOGIES_FILE = REPO_ROOT / "dataset" / "analogies" / "bats_analogies.txt"

WORK_ROOT = (Path("/content") if Path("/content").exists() else REPO_ROOT) / "entity_swap_r0_ablation"
STAGED_DIR = WORK_ROOT / "numeric_ours"
OUTPUT_ROOT = WORK_ROOT / "outputs"
MODES = ["full", "suppress", "inject"]

assert GRAPH_SRC_DIR.is_dir(), f"missing labeled graph dir: {GRAPH_SRC_DIR}"
assert ANALOGIES_FILE.exists(), f"missing analogies file: {ANALOGIES_FILE}"
print("relation:", RELATION_IDX, RELATION_NAME)
print("graph src:", GRAPH_SRC_DIR)
print("device:", DEVICE)

In [ ]:
# Stage NNN.sng.pt symlinks; the eval requires the .sng.pt suffix and exactly 100 numeric files.
from summarization.summarize import SummaryGraph

if STAGED_DIR.exists():
    shutil.rmtree(STAGED_DIR)
STAGED_DIR.mkdir(parents=True)

for idx in range(100):
    src = GRAPH_SRC_DIR / f"{idx:03d}.pt"
    assert src.exists(), f"missing labeled graph {src}"
    (STAGED_DIR / f"{idx:03d}.sng.pt").symlink_to(src)

staged = sorted(STAGED_DIR.glob("[0-9][0-9][0-9].sng.pt"))
assert len(staged) == 100, f"expected 100 staged graphs, found {len(staged)}"

probe = SummaryGraph.load(str(STAGED_DIR / "000.sng.pt"))
assert isinstance(probe.metadata.get("prompt"), str) and probe.metadata["prompt"], "000 missing metadata['prompt']"
assert probe.metadata.get("prompt_tokens"), "000 missing metadata['prompt_tokens']"
print("staged", len(staged), "graphs at", STAGED_DIR)
print("probe prompt:", probe.metadata["prompt"])

In [ ]:
import argparse

from circuit_tracer import ReplacementModel
from eval.eval_entity_swap import run_entity_swap

print("loading replacement model once:", MODEL_NAME, "/", TRANSCODER_SET)
model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_SET,
    backend=BACKEND,
    lazy_encoder=True,
    dtype=DTYPE_MAP[DTYPE],
    device=torch.device(DEVICE) if DEVICE else None,
)


def run_mode(mode: str) -> Path:
    output_dir = OUTPUT_ROOT / mode
    args = argparse.Namespace(
        graph_dir=STAGED_DIR,
        analogies_file=ANALOGIES_FILE,
        negation_coefficients=NEG_COEFFS,
        addition_coefficients=ADD_COEFFS,
        relations=str(RELATION_IDX),
        sample_pairs_per_relation=SAMPLE_PAIRS_PER_RELATION,
        pair_list=None,
        random_state=RANDOM_STATE,
        output_dir=output_dir,
        layers_below=LAYERS_BELOW,
        layers_above=LAYERS_ABOVE,
        mode=mode,
    )
    print(f"=== running entity swap: mode={mode} ===", flush=True)
    run_entity_swap(model, args)
    return output_dir


output_dirs = {mode: run_mode(mode) for mode in MODES}
print("outputs:", {mode: str(path) for mode, path in output_dirs.items()})

In [ ]:
import pandas as pd

# Same metric set as the prior runs (thesis table + kmeans notebook), keyed by (condition, donor_factor).
frames = []
for mode in MODES:
    summary = pd.read_csv(output_dirs[mode] / "swap_summary.csv")
    summary.insert(0, "condition", mode)
    frames.append(summary)

summary_df = pd.concat(frames, ignore_index=True)
summary_df["delta_p_source"] = summary_df["mean_p_source_steered"] - summary_df["mean_p_source_clean"]
summary_df["delta_p_donor"] = summary_df["mean_p_donor_steered"] - summary_df["mean_p_donor_clean"]

report_cols = [
    "condition",
    "donor_factor",
    "n_attempted",
    "n_eligible",
    "top1_hit_rate",
    "top1_hit_exact_rate",
    "top5_hit_rate",
    "success_rate",
    "eligible_success_rate",
    "mean_p_source_clean",
    "mean_p_source_steered",
    "delta_p_source",
    "mean_p_donor_clean",
    "mean_p_donor_steered",
    "delta_p_donor",
]
report_df = summary_df.sort_values(["condition", "donor_factor"]).reset_index(drop=True)[report_cols]
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
display(report_df)

In [ ]:
# Write a self-contained HTML report so it can be downloaded from Colab.
rate_cols = [
    "top1_hit_rate",
    "top1_hit_exact_rate",
    "top5_hit_rate",
    "success_rate",
    "eligible_success_rate",
    "mean_p_source_clean",
    "mean_p_source_steered",
    "delta_p_source",
    "mean_p_donor_clean",
    "mean_p_donor_steered",
    "delta_p_donor",
]
fmt = {col: "{:.4f}".format for col in rate_cols}
table_html = report_df.to_html(index=False, formatters=fmt, border=0)

report_path = OUTPUT_ROOT / "report.html"
report_path.write_text(
    """<!doctype html>
<html><head><meta charset=\"utf-8\"><title>Entity-swap relation 0 ablation</title>
<style>
body{font-family:system-ui,Arial,sans-serif;margin:2rem;color:#1a1a1a;}
h1{font-size:1.4rem;}h2{font-size:1.05rem;margin-top:1.5rem;}
table{border-collapse:collapse;margin-top:.5rem;font-size:.85rem;}
th,td{border:1px solid #ccc;padding:4px 8px;text-align:right;}
th{background:#f2f2f2;}td:first-child,th:first-child{text-align:left;}
code{background:#f2f2f2;padding:1px 4px;border-radius:3px;}
</style></head><body>
<h1>Entity-swap relation 0 (capital_country): full vs suppress-only vs inject-only</h1>
<p>Source Output features suppressed with factor <code>-2</code>; donor Output features injected at the listed
positive factor. Conditions share the same %d sampled source&rarr;donor pairs (random_state=%d).
Transcoder set: <code>%s</code>. &Delta;p = steered &minus; clean.</p>
%s
</body></html>
"""
    % (SAMPLE_PAIRS_PER_RELATION, RANDOM_STATE, TRANSCODER_SET, table_html),
    encoding="utf-8",
)
print("wrote report:", report_path)

In [ ]:
# Verify: same 50 pairs across conditions; full/inject have 3 donor factors, suppress has 1.
pair_sets = {}
factor_counts = {}
for mode in MODES:
    results = pd.read_csv(output_dirs[mode] / "swap_results.csv")
    pairs = set(map(tuple, results[["source_idx", "donor_idx"]].drop_duplicates().to_numpy().tolist()))
    pair_sets[mode] = pairs
    factor_counts[mode] = results["donor_factor"].nunique()
    print(f"{mode}: {len(pairs)} pairs, {factor_counts[mode]} donor factor(s), {len(results)} rows")

assert pair_sets["full"] == pair_sets["suppress"] == pair_sets["inject"], "pair sets differ across conditions"
assert all(len(p) == SAMPLE_PAIRS_PER_RELATION for p in pair_sets.values()), "expected 50 pairs per condition"
assert factor_counts["full"] == 3 and factor_counts["inject"] == 3, "full/inject should have 3 donor factors"
assert factor_counts["suppress"] == 1, "suppress should have a single (0.0) donor factor"

n_eligible = {mode: int(summary_df.loc[summary_df.condition == mode, "n_eligible"].iloc[0]) for mode in MODES}
assert len(set(n_eligible.values())) == 1, f"n_eligible differs across conditions: {n_eligible}"
print("verification passed; n_eligible =", n_eligible)